# GRPO reward-scaling experiment: Colab A100 runner

One-click pipeline for the AISTATS LLM experiment: Qwen2.5-0.5B-Instruct + LoRA, TRL
`GRPOTrainer`, comparing GRPO / Dr.GRPO / global normalization reward scaling under a
**conflicting-preference task**. Every training example asks the SAME question ("pick
an integer between 0 and 100"), but carries a hidden family label (never shown to the
model): family A wants the answer near 80, family B wants it near 20, and B's reward
is scaled by `k`. A and B pull the SAME output distribution in opposite directions, so
`k` should visibly shift where the policy settles -- unless an estimator's own reward
normalization (GRPO's per-instance std) cancels that shift out.

**Run cells top to bottom.** There is no separately gated pilot step: all 7 configs
run directly, seed 0 first, with a pass/fail check printed right after seed 0 so you
can stop early if the design isn't working before spending seeds 1-2's GPU time.
Target: ~1.5 hours total on one A100 (short, 8-token completions; 80 steps/run).

All results are saved to Google Drive (`grpo_llm_results/`) immediately after each
run finishes, so the sweep can resume from a Colab disconnect without losing completed
work.

## 1. Check for a GPU

Sets `PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True` before torch ever loads --
it reduces allocator fragmentation under GRPO's bursty generate-then-backward memory
pattern, and only takes effect if set prior to CUDA context init.

In [ ]:
import os

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

import subprocess

try:
    import torch
    has_gpu = torch.cuda.is_available()
except ImportError:
    has_gpu = False

if not has_gpu:
    raise RuntimeError(
        'No GPU detected. Go to Runtime > Change runtime type, select a GPU '
        '(A100 recommended), then Runtime > Restart session and run all cells again.'
    )

print(subprocess.run(['nvidia-smi'], capture_output=True, text=True).stdout)

## 2. Config -- edit if your fork/repo name differs

In [ ]:
GITHUB_USER = "nilay47"
GITHUB_REPO = "agenthon"
DRIVE_RESULTS_DIR = "/content/drive/MyDrive/grpo_llm_results"
REPO_DIR = f"/content/{GITHUB_REPO}"

## 3. Mount Google Drive

Also points the HuggingFace cache at Drive, so the ~1GB model download survives a
disconnect instead of being wiped along with the rest of the Colab VM's disk.

In [ ]:
from google.colab import drive
import os

drive.mount('/content/drive')
os.makedirs(DRIVE_RESULTS_DIR, exist_ok=True)
print(f"Results will be saved to {DRIVE_RESULTS_DIR}")

os.environ["HF_HOME"] = "/content/drive/MyDrive/hf_cache"
os.makedirs(os.environ["HF_HOME"], exist_ok=True)
hf_home = os.environ['HF_HOME']
print(f"HF cache: {hf_home}")

## 4. Clone the repo

If the repo is **private**, add a `GITHUB_TOKEN` Colab secret first (key icon in the
left sidebar) with a personal access token that has `repo` scope, and grant this
notebook access to it when prompted. If the repo is public, this works with no token.

In [ ]:
import os

token = None
try:
    from google.colab import userdata
    token = userdata.get('GITHUB_TOKEN')
except Exception:
    pass

if token:
    clone_url = f"https://{token}@github.com/{GITHUB_USER}/{GITHUB_REPO}.git"
else:
    clone_url = f"https://github.com/{GITHUB_USER}/{GITHUB_REPO}.git"

if not os.path.exists(REPO_DIR):
    !git clone {clone_url} {REPO_DIR}
else:
    print(f"{REPO_DIR} already exists, skipping clone (pulling latest instead)")
    !cd {REPO_DIR} && git pull

%cd {REPO_DIR}/llm

## 5. Install dependencies (Colab-safe: never touches torch/CUDA)

Installs `trl`/`peft`/`transformers`/`accelerate`/`datasets` with `--no-deps`, so pip
never resolves THEIR torch dependency against Colab's preinstalled build -- that
resolution is what previously pulled in a mismatched torch/CUDA wheel and broke
`trl.trainer.grpo_trainer` with `libcudart.so.13: cannot open shared object file`.
The lightweight, pure-Python packages those five need are installed separately with
normal resolution (they can't touch torch/CUDA either way). vLLM is intentionally
**not** installed -- `run.py` already falls back to plain HF generation automatically.

Colab also preinstalls an old `torchao` that `peft` refuses to import against
(`Found an incompatible version of torchao`) -- we never use it, so it's removed.

In [ ]:
!pip install -q --no-deps -r requirements-colab.txt
!pip install -q huggingface_hub tokenizers safetensors regex pyyaml filelock requests \
    tqdm packaging psutil pandas pyarrow dill multiprocess fsspec xxhash scipy
!pip uninstall -y -q torchao

import importlib.metadata as _metadata

for _pkg in ['torch', 'transformers', 'trl', 'peft', 'accelerate', 'datasets']:
    try:
        _ver = _metadata.version(_pkg)
    except _metadata.PackageNotFoundError:
        _ver = 'NOT INSTALLED'
    print(f"{_pkg} == {_ver}")

import torch

print(f"torch.__version__ = {torch.__version__}")
print(f"torch.version.cuda = {torch.version.cuda}")
print(f"torch.cuda.is_available() = {torch.cuda.is_available()}")

try:
    import trl.trainer.grpo_trainer  # noqa: F401
    from peft import LoraConfig, get_peft_model  # noqa: F401
    print("trl.trainer.grpo_trainer and peft imported OK")
except Exception as e:
    raise RuntimeError(
        f"Import check failed against torch {torch.__version__} (CUDA {torch.version.cuda}). "
        f"This usually means a pip install pulled in a mismatched torch/CUDA build, or an "
        f"incompatible optional dependency (like torchao) is still installed. "
        f"Original error: {e!r}"
    ) from e

## 5b. PREFLIGHT: exercise every training config on 2 tiny steps

Loads the real model + LoRA and runs 2 GRPO steps (2 prompts x G=4, 8-token
completions -- the real `max_completion_length`) for **each** of the four training
configs used later -- `scale_rewards` `"group"`, `"batch"`, `"none"`, and the
sigma-sampling method -- including one eval-callback call and one save to the Drive
`preflight/` subfolder each. This is the exact same code path `run.py` uses for the
real runs, just at a scale that finishes in seconds, so a config-level break (like the
torchao import failure above) surfaces here instead of after committing GPU time.

In [ ]:
import os
import sys
sys.path.insert(0, '.')
import config
from run import run_one

PREFLIGHT_DIR = os.path.join(DRIVE_RESULTS_DIR, 'preflight')

for _method in ['grpo', 'global', 'drgrpo', 'sigma_sampling']:
    print(f'--- preflight: {_method} ---')
    try:
        run_one(_method, k=1, seed=0, max_steps=2, out_dir=PREFLIGHT_DIR,
                prompts_per_step=2, g=4, max_completion_length=config.MAX_COMPLETION_LENGTH,
                eval_every=1)
    except Exception as e:
        raise RuntimeError(
            f"PREFLIGHT FAILED for config '{_method}'. Original error: {e!r}"
        ) from e

print("PREFLIGHT PASSED")

## 5c. PREFLIGHT: peak GPU memory at the REAL batch shape

The config-smoke preflight above uses a tiny 2-prompt batch, which does not stress
memory. This one runs the **actual** training shape (16 prompts x G=8 = 128
completions per optimizer step, full batch in one backward -- no micro-batching, since
8-token completions keep `grad_logits` small enough not to need it) for one config,
with completions forced out to the full `max_completion_length` via `min_new_tokens`
(the true worst case), and reports peak GPU memory. Should now trivially pass (expect
well under 2 GB) -- kept as a safety net in case `config.MAX_COMPLETION_LENGTH` or
`config.MICRO_BATCH_COMPLETIONS` is ever changed back toward the old, longer-completion
regime. Fails if peak reserved memory exceeds 34 GB (A100 is 40GB).

In [ ]:
import torch

torch.cuda.reset_peak_memory_stats()
run_one('grpo', k=1, seed=0, max_steps=1, out_dir=PREFLIGHT_DIR,
        prompts_per_step=16, g=8, max_completion_length=config.MAX_COMPLETION_LENGTH, eval_every=1,
        extra_grpo_kwargs=dict(generation_kwargs=dict(min_new_tokens=config.MAX_COMPLETION_LENGTH)))

peak_allocated_gb = torch.cuda.max_memory_allocated() / 1024**3
peak_reserved_gb = torch.cuda.max_memory_reserved() / 1024**3
print(f"peak memory allocated: {peak_allocated_gb:.2f} GB")
print(f"peak memory reserved:  {peak_reserved_gb:.2f} GB")

if peak_reserved_gb > 34:
    current_micro = config.MICRO_BATCH_COMPLETIONS
    raise RuntimeError(
        f"Peak GPU memory ({peak_reserved_gb:.2f} GB reserved) exceeds the 34 GB safety "
        f"threshold for a 40GB A100. Reduce config.MICRO_BATCH_COMPLETIONS (currently "
        f"{current_micro}) and rerun this cell."
    )
print("MEMORY PREFLIGHT PASSED")

## 6. Baseline sanity check

Runs the **untrained** model on 100 fresh prompts: reports the initial distribution
of the parsed integer u (mean, median, 10-bin histogram over [0,100]) and parse rate,
plus an informational within-group-std-of-u check at training temperature (not a hard
gate -- just a flag if there's too little output variety for GRPO to have any signal).

In [ ]:
!python baseline_eval.py --out_dir "{DRIVE_RESULTS_DIR}"

import json, os
with open(os.path.join(DRIVE_RESULTS_DIR, "baseline_eval.json")) as f:
    baseline_result = json.load(f)
print(baseline_result)

## 7. Determinism check (GRPO advantages must be k-invariant)

GRPO's per-group reward normalization, `(r - mean(r)) / (std(r) + eps)`, is an exact
invariant of a PURE rescaling `r -> k*r` -- the mechanism behind GRPO being predicted
to erase k's effect. This runs GRPO k=1 and GRPO k=10 for ONE step each, same seed
(`run_one` reseeds torch before building the model, so both calls generate identical
completions), and checks the first step's advantages match to within TRL's ~1e-4
epsilon. Quick (two 1-step runs) -- confirms the mechanism before spending GPU time
on full reruns below.

In [ ]:
import sys
sys.path.insert(0, '.')
from check_determinism import run_check

determinism_passed, determinism_max_diff = run_check(seed=0)

## 8. Targeted rerun: GRPO (all 3 seeds) + other configs (seeds 1-2)

**Why GRPO needs a full redo:** an earlier bug left the unparsable-completion
penalty fixed at -1 for family B instead of scaling it by k (-k), which broke the
exact k-invariance GRPO depends on whenever parse rate < 1 (the generic case). Any
`grpo_k*_seed*.json` already on Drive predates the fix and is deleted here before
rerunning GRPO for all 3 seeds. Dr.GRPO/Global/sigma-sampling were not relying on
that invariance (their qualitative behavior isn't predicted to depend on it), so
their existing seed-0 results are kept -- only seeds 1-2 are added for them.

In [ ]:
import os
from full import run_sweep, result_path

for _k in [1, 10]:
    for _seed in [0, 1, 2]:
        _path = result_path(DRIVE_RESULTS_DIR, 'grpo', _k, _seed)
        if os.path.exists(_path):
            print('deleting stale (pre-fix):', _path)
            os.remove(_path)

grpo_results = run_sweep([0, 1, 2], DRIVE_RESULTS_DIR, skip_existing=True,
                          configs=[('grpo', 1), ('grpo', 10)])

In [ ]:
from full import CONFIGS

other_configs = [c for c in CONFIGS if c[0] != 'grpo']
other_results = run_sweep([1, 2], DRIVE_RESULTS_DIR, skip_existing=True, configs=other_configs)

## 9. Seed-aware pass check

(1) GRPO: k=10's mean final mean-u must fall within k=1's OWN 95% seed-to-seed CI --
i.e. indistinguishable from k=1's natural seed variability (not just "close" by a
fixed threshold). (2) Dr.GRPO: the paired (per-seed) shift (k=1 minus k=10) must have
a 95% CI that excludes zero -- the shift toward B's target must survive seed noise,
not just appear in one lucky seed.

In [ ]:
import json, os
from pilot import check_pass

# Load every (config, seed) that should exist on Drive by now -- seed 0 for the
# non-GRPO configs isn't in grpo_results/other_results (those only cover what was
# just (re)run), so load it directly here. Together with grpo_results/other_results
# this gives a COMPLETE all_results covering all 7 configs x all 3 seeds, for both
# the pass-check below and the final summary (section 10).
all_results = {}
for _method, _k in [c for c in CONFIGS if c[0] != 'grpo']:
    _key = f'{_method}_k{_k}_seed0'
    _path = result_path(DRIVE_RESULTS_DIR, _method, _k, 0)
    with open(_path) as f:
        all_results[_key] = json.load(f)
all_results.update(grpo_results)
all_results.update(other_results)

results_for_check = {
    'grpo_k1': [all_results[f'grpo_k1_seed{s}'] for s in [0, 1, 2]],
    'grpo_k10': [all_results[f'grpo_k10_seed{s}'] for s in [0, 1, 2]],
    'drgrpo_k1': [all_results[f'drgrpo_k1_seed{s}'] for s in [0, 1, 2]],
    'drgrpo_k10': [all_results[f'drgrpo_k10_seed{s}'] for s in [0, 1, 2]],
}
passed, detail, final_u = check_pass(results_for_check)

print(detail)
print('SEED-AWARE CHECK', 'PASSED' if passed else 'FAILED')

with open(os.path.join(DRIVE_RESULTS_DIR, "pilot_summary.json"), "w") as f:
    json.dump(dict(passed=passed, detail=detail, final_mean_u=final_u,
                   determinism_passed=determinism_passed,
                   determinism_max_diff=determinism_max_diff), f, indent=2)

## 10. Final summary: table + figure

Table: final (pooled) mean u per config, mean +- 95% CI across the 3 seeds. Figure:
mean u over training steps, one panel per method that has both k values (Dr.GRPO /
GRPO / Global norm) -- k=1 solid, k=10 dashed -- with dotted horizontal lines at the
predicted mean u (GRPO ~50 for both k; Dr.GRPO ~50 at k=1, ~25 at k=10).

In [ ]:
import os
from full import build_summary
from summarize import print_summary_table, make_figure

summary = build_summary(all_results, [0, 1, 2])
print_summary_table(summary)
fig_paths = make_figure(summary, os.path.join(DRIVE_RESULTS_DIR, 'summary_figure'))
print("wrote", fig_paths)

import json
with open(os.path.join(DRIVE_RESULTS_DIR, "full_summary.json"), "w") as f:
    json.dump(summary, f, indent=2)